In [1]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 121.8 MB/s eta 0:00:00


# 1. Loading the document

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [3]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

1325

# 2. Spliting document into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [9]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [11]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="your_api_key_here"
)

# 6. Building Prompts and chains

In [36]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [31]:
def ask(query):
    top_docs = faiss_db.similarity_search(query, k=10)
    prompt_template = PromptTemplate.from_template(
      "Give answer according to the following passages in the quran {context}"
      "If the answer is not present in the given context then give answer according to the internet sources."
      "But do inform that there are no passages in the quran about the question."
      "Answer the following question {question}."
    )
    chain = prompt_template | llm | StrOutputParser()
    result = chain.invoke({"context": top_docs, "question": query})
    return result

Relevant questions

In [33]:
asnwer = ask("What does the Quran say about Day of Judgment?")
print(asnwer)

The Quran has a significant amount of information about the Day of Judgment, which is also known as the Day of Reckoning or Qiyamah. Here are some key points mentioned in the Quran:

1. **The Day of Judgment is inevitable**: The Quran emphasizes that the Day of Judgment is a certainty and will inevitably come to pass. (Surah 3:185, Surah 6:73, Surah 14:48)
2. **All souls will be accountable**: On the Day of Judgment, every soul will be held accountable for their deeds, and no one will be able to escape judgment. (Surah 3:25-30, Surah 82:14-19)
3. **The Record of Deeds**: Every individual's deeds, good or bad, will be recorded in a book, and they will be judged based on their actions. (Surah 17:13-14, Surah 54:52-53)
4. **The Balance of Justice**: On the Day of Judgment, a balance of justice will be set up, and every person's deeds will be weighed. Those whose good deeds outweigh their bad deeds will be rewarded, and those whose bad deeds outweigh their good deeds will be punished. (Sur

In [37]:
asnwer = ask("What of someone donot FAST in the month of RAMADAN?")
print(asnwer)

According to the Quran and Islamic teachings, fasting during the month of Ramadan is one of the Five Pillars of Islam and is obligatory for all adult Muslims who are physically and mentally able. 

If someone does not fast during the month of Ramadan without a valid reason, it is considered a serious sin. The Quran states:

"Oh you who believe, fasting is prescribed for you as it was prescribed for those before you, that you may become righteous." (Surah Al-Baqarah, 2:183)

The Prophet Muhammad (peace be upon him) also said:

"Islam is built on five pillars: the testimony that there is no god but Allah and that Muhammad is the Messenger of Allah, the establishment of prayer, the payment of zakat, the pilgrimage to the House, and the fasting of Ramadan." (Sahih Bukhari)

If someone is unable to fast due to a valid reason, such as illness, travel, or menstruation, they are exempt from fasting, but they must make up the missed days later.

If someone intentionally breaks their fast withou

Irrelevant questions

In [38]:
asnwer = ask("When did dinosaurs came into being?")
print(asnwer)

There are no passages in the Quran about the question of when dinosaurs came into being. However, according to internet sources and scientific research, dinosaurs are believed to have originated during the Middle to Late Triassic period, around 230-245 million years ago. They dominated Earth's landscapes during the Mesozoic Era, which lasted from about 252 million to 66 million years ago. The exact timing and details of their emergence are still the subject of ongoing scientific study and research.
